# GRPO for multi-turn knowledge graph navigation on one Kaggle T4

This notebook uses **real graph tools and Hugging Face TRL's actual GRPOTrainer**. It does not implement SFT or a substitute policy-gradient algorithm. Qwen3 samples tool-call tokens, graph observations enter subsequent generations, and only model-generated tokens contribute to the policy loss. The sole reward is membership of the **actual submitted ID** in the private complete answer set.

**Saved development run:** CPU preflight only. GPU-dependent results are absent unless this notebook is executed with the selected CUDA stack. No pretrained-model rollout, successful optimization step, T4 memory measurement, or adapter reload is claimed from the CPU outputs.

**Kaggle quick start**
1. Create a Python notebook, enable **one T4** GPU and internet for package/model downloads.
2. Attach the repository as a Kaggle dataset, including `src/kg_rlvr/`, `requirements-grpo.txt`, and the existing `data/graph`, `data/rl`, and `data/metadata` files. A separately mounted data directory is also supported.
3. Configure the paths and switches below. For a fresh runtime with different packages, set `INSTALL_DEPENDENCIES=True` and run the setup cell. Restart the session if instructed, then set it back to `False` and run all cells.
4. The default GPU run evaluates the base model and attempts **two genuine GRPO smoke steps**. Full 30-step training is explicit opt-in: `RUN_FULL_TRAINING=True`.
5. Keep outputs under `/kaggle/working/`. Save a Kaggle notebook version **with outputs** or download the ZIP to retain files after the session ends.

**Contents:** [1 Setup](#setup) · [2 Hardware](#hardware) · [3 Data](#data) · [4 Tools](#tools) · [5 Model](#model) · [6 Baseline](#baseline) · [7 GRPO](#grpo) · [8 Smoke](#smoke) · [9 Full training](#full) · [10 Metrics](#metrics) · [11 Evaluation](#evaluation) · [12 Export](#export) · [13 Trajectories and evidence](#trajectories)

<a id="setup"></a>
## 1. Environment setup and pinned dependencies

All user choices are collected here. `DATA_ROOT` is not tied to a Kaggle slug. Inputs are read-only; checkpoints, logs, adapters, and figures are written elsewhere. CPU execution runs the non-training checks and explicitly skips model phases.

The selected stack is Torch **2.6.0/cu124**, Transformers **5.3.0**, TRL **1.8.0**, PEFT **0.18.1**, Accelerate **1.10.1**, bitsandbytes **0.49.2**, datasets **4.8.3**. It was import/API checked on CPU; the CUDA/QLoRA combination still requires the smoke test below. NumPy **1.26.4**, Pandas **2.2.3** and PyArrow **21.0.0** are pinned together to avoid binary-version mismatches. Target Python is 3.11/3.12.

Sources: [released TRL documentation](https://huggingface.co/docs/trl/v1.8.0/en/grpo_trainer), [TRL 1.8.0 implementation](https://github.com/huggingface/trl/blob/v1.8.0/trl/trainer/grpo_trainer.py), [Transformers 5.3.0](https://huggingface.co/docs/transformers/v5.3.0/en/index), [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization), [bnb 0.49.2 supported builds](https://huggingface.co/docs/bitsandbytes/v0.49.2/en/installation), [official Torch CUDA wheels](https://pytorch.org/get-started/previous-versions/).

No vLLM, FlashAttention, reward model, LLM judge, or sentencepiece installation is needed. Qwen3 uses its BPE tokenizer and native tool-call chat template. Setup optionally removes unused audio/vision packages from the **fresh Kaggle session** to avoid their compiled extensions conflicting with the pinned Torch version. If a changed library was already imported, restart before continuing; package installation is not a training smoke test.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata, tempfile

# PATHS: edit the overrides for your attached Kaggle dataset.
PROJECT_ROOT_OVERRIDE = None  # e.g. Path("/kaggle/input/my-project/graph_rlvr")
DATA_ROOT_OVERRIDE = None     # e.g. Path("/kaggle/input/my-kg-dataset/data")
WORK_ROOT_OVERRIDE = None     # None -> /kaggle/working on Kaggle; local artifacts/grpo otherwise

# EXECUTION: a GPU smoke run is enabled; full training is opt-in.
INSTALL_DEPENDENCIES = False
REMOVE_UNUSED_MULTIMODAL_PACKAGES = True
RUN_GPU = True
RUN_BASELINE = True
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = False
RUN_NAME = "kg_grpo_seed41"
SELECTED_GPU = "0"  # expose one device BEFORE importing torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MODEL_REVISION_OVERRIDE = None  # immutable known revision selected below
TRAIN_HOPS = [2, 3]            # [2], [3], [4], or any combination is supported
EVAL_HOPS = [2, 3, 4]
MAX_TRAIN_EXAMPLES = 1000       # limit, not a request to invent additional rows
BASELINE_EVAL_SIZE = 30
TEST_EVAL_SIZE = 30
RELOAD_EVAL_SIZE = 3
SEED = 41

NUM_GENERATIONS = 4
LORA_R, LORA_ALPHA = 8, 16
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-6
MAX_TOOL_CALLS = 8
MAX_TURN_TOKENS = 128
MAX_COMPLETION_TOKENS = 1536    # all model + observation tokens after the initial prompt
MAX_PROMPT_TOKENS = 1024
MAX_CONTEXT_TOKENS = 3072
MAX_NEIGHBORS_PER_RESPONSE = 16
MAX_OBSERVATION_CHARS = 4000
SMOKE_TEST_STEPS = 2
INITIAL_TRAINING_STEPS = 30
SAVE_STEPS = 10
EVALUATE_EVERY_STEPS = 0        # >0 enables measured greedy validation during full training
PERIODIC_EVAL_SIZE = 6
RESUME_FROM_CHECKPOINT = None   # full checkpoint-* folder, not an adapter-only export
MAKE_ZIP = True

def find_project():
    if PROJECT_ROOT_OVERRIDE:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src/kg_rlvr/graph_index.py").exists():
            return parent
    candidates = list(Path("/kaggle/input").glob("*/src/kg_rlvr/graph_index.py"))
    candidates += list(Path("/kaggle/input").glob("*/*/src/kg_rlvr/graph_index.py"))
    if len(candidates) == 1:
        return candidates[0].parents[2]
    raise FileNotFoundError("Set PROJECT_ROOT_OVERRIDE to the attached repository containing src/kg_rlvr.")

PROJECT_ROOT = find_project()
DATA_ROOT = Path(DATA_ROOT_OVERRIDE or PROJECT_ROOT / "data").expanduser().resolve()
IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
WORK_ROOT = Path(WORK_ROOT_OVERRIDE or ("/kaggle/working" if IS_KAGGLE else PROJECT_ROOT / "artifacts/grpo")).expanduser().resolve()
if WORK_ROOT == DATA_ROOT or DATA_ROOT in WORK_ROOT.parents:
    raise ValueError("Working outputs must be outside the source dataset")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be a simple directory name")
if GRADIENT_ACCUMULATION_STEPS < NUM_GENERATIONS or GRADIENT_ACCUMULATION_STEPS % NUM_GENERATIONS:
    raise ValueError("Microbatch 1 requires accumulation divisible by NUM_GENERATIONS")
if NUM_GENERATIONS < 2:
    raise ValueError("GRPO requires at least two generations per prompt")
if set(TRAIN_HOPS) - {2,3,4} or not TRAIN_HOPS or set(EVAL_HOPS) - {2,3,4} or not EVAL_HOPS:
    raise ValueError("Choose hop categories from 2, 3, 4")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CUDA_VISIBLE_DEVICES"] = SELECTED_GPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "kg_grpo_matplotlib"))
os.environ.setdefault("HF_HOME", str(WORK_ROOT / "hf_cache"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT = WORK_ROOT / "runs" / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints" / RUN_NAME
EVALUATION_DIR = RUN_ROOT / "evaluation"
ADAPTER_DIR = WORK_ROOT / "grpo_adapter"
print(f"Project: {PROJECT_ROOT}\nData: {DATA_ROOT}\nWorking outputs: {WORK_ROOT}\nSingle GPU selection: {SELECTED_GPU}")

Project: /Users/alirezaak/Documents/research/graph_rlvr
Data: /Users/alirezaak/Documents/research/graph_rlvr/data
Working outputs: /Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo
Single GPU selection: 0


In [2]:
if INSTALL_DEPENDENCIES:
    if not IS_KAGGLE:
        raise RuntimeError("Automatic CUDA installation is scoped to a fresh Kaggle session. For local CPU checks use requirements.txt; inspect requirements-grpo.txt separately.")
    loaded = {name for name in ["torch", "numpy", "pandas", "transformers", "trl", "peft", "accelerate", "datasets", "pyarrow"] if name in sys.modules}
    if REMOVE_UNUSED_MULTIMODAL_PACKAGES:
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "torchcodec", "librosa"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-r", str(PROJECT_ROOT / "requirements-grpo.txt")])
    if loaded:
        raise RuntimeError(f"Setup changed libraries already imported ({sorted(loaded)}). Restart the Kaggle session, set INSTALL_DEPENDENCIES=False, then Run All. Do not continue with mixed in-memory versions.")
from kg_rlvr.training.grpo import dependency_versions, MODEL_REVISIONS, PINS
MODEL_REVISION = MODEL_REVISION_OVERRIDE or MODEL_REVISIONS.get(MODEL_NAME)
if not MODEL_REVISION:
    raise ValueError("Supply an explicit immutable revision for a different model")
print("Requested pins:", json.dumps(PINS, indent=2))
print("Installed:", json.dumps(dependency_versions(), indent=2))
print("Selected model/revision:", MODEL_NAME, MODEL_REVISION)

Requested pins: {
  "torch": "2.6.0",
  "transformers": "5.3.0",
  "trl": "1.8.0",
  "peft": "0.18.1",
  "accelerate": "1.10.1",
  "bitsandbytes": "0.49.2",
  "datasets": "4.8.3",
  "jmespath": "1.0.1",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "pyarrow": "21.0.0"
}
Installed: {
  "torch": "2.14.0",
  "transformers": "4.48.1",
  "trl": null,
  "peft": null,
  "accelerate": "1.10.0",
  "bitsandbytes": null,
  "datasets": "4.4.1",
  "jmespath": "0.10.0",
  "numpy": "1.26.4",
  "pandas": "2.3.2",
  "pyarrow": "22.0.0",
  "tokenizers": "0.21.0",
  "huggingface-hub": "0.36.2",
  "matplotlib": "3.7.2",
  "psutil": "5.9.0"
}
Selected model/revision: Qwen/Qwen3-1.7B 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e


<a id="hardware"></a>
## 2. GPU, CUDA, VRAM, and system memory

Only the selected GPU is visible to Torch; the `nvidia-smi` inventory can still show all physical devices. Two T4s do not provide a pooled 32 GB device for this notebook. The T4 path uses FP16 computation because it lacks native BF16 support; the runtime capability check selects BF16 only when supported. Training uses microbatch 1, gradient checkpointing, frozen NF4 base weights, and trainable LoRA parameters.

In [3]:
import gc, math, random, platform, time
import warnings
with warnings.catch_warnings(record=True) as dependency_notices:
    warnings.simplefilter("always", UserWarning)
    import pandas as pd
import torch
import psutil
from IPython.display import display, Markdown, HTML
from kg_rlvr.training.grpo import memory_snapshot, fingerprint_inputs, native_bf16_supported
from kg_rlvr.training.environment import EpisodeLimits, load_tasks, public_prompt, GraphEnvironment, TOOL_SCHEMAS

try:
    inventory = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"], text=True, capture_output=True, check=False)
    print(inventory.stdout.strip() or "nvidia-smi inventory unavailable")
except FileNotFoundError:
    print("nvidia-smi unavailable on this machine")
HAS_CUDA = torch.cuda.is_available()
RUN_MODEL = RUN_GPU and HAS_CUDA
hardware = {"python": platform.python_version(), "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
            "visible_cuda_devices": torch.cuda.device_count(), "system_available_gib": psutil.virtual_memory().available / 2**30}
if HAS_CUDA:
    properties = torch.cuda.get_device_properties(0)
    hardware.update(gpu_name=properties.name, gpu_vram_gib=properties.total_memory / 2**30,
                    gpu_capability=torch.cuda.get_device_capability(0), bf16_native_supported=native_bf16_supported())
display(pd.Series(hardware, name="runtime hardware").to_frame())
if dependency_notices:
    display(pd.DataFrame({"environment notice": [str(w.message) for w in dependency_notices]}))
MEMORY = [memory_snapshot("before_model")]
if not RUN_MODEL:
    display(Markdown("**CPU preflight mode:** model downloads, inference, training, GPU memory measurements, and adapter save/reload are skipped. Subsequent skipped sections do not count as successful GPU tests."))
LIMITS = EpisodeLimits(MAX_TOOL_CALLS, MAX_NEIGHBORS_PER_RESPONSE, MAX_OBSERVATION_CHARS,
                      MAX_TURN_TOKENS, MAX_COMPLETION_TOKENS, MAX_PROMPT_TOKENS, MAX_CONTEXT_TOKENS)
random.seed(SEED)
torch.manual_seed(SEED)

nvidia-smi unavailable on this machine


,runtime hardware
python,3.11.5
torch,2.14.0
cuda_runtime,None
visible_cuda_devices,0
system_available_gib,0.755997


,environment notice
0,Pandas requires version '1.3.6' or newer of 'b...


**CPU preflight mode:** model downloads, inference, training, GPU memory measurements, and adapter save/reload are skipped. Subsequent skipped sections do not count as successful GPU tests.

<a id="data"></a>
## 3. Dataset loading, validation, and model/verifier separation

The existing GraphIndex and in-memory validator are reused. Both source files and existing reports are fingerprinted. The training dataset contains only a `prompt` column; complete answers, programs and reference paths stay in the private TaskStore. Evaluation IDs are fixed, balanced across selected hop categories, and deduplicated by query instance.

The policy selects **one** correct entity. This task intentionally differs from the earlier complete-set conversational export. Reward is exactly `float(actual_submitted_id in complete_gold_answer_ids)`, with 0 for wrong or absent submissions. No reward shaping, efficiency bonus, progress score, or path-matching reward is used.

In [4]:
from collections import Counter
from dataclasses import asdict
INPUT_SHA256 = fingerprint_inputs(DATA_ROOT)
graph, store, dataset_validation = load_tasks(DATA_ROOT, max_tool_calls=MAX_TOOL_CALLS, validate=True)
train_ids = store.select("train", TRAIN_HOPS, MAX_TRAIN_EXAMPLES, SEED, unique_instances=True)
validation_ids = store.select("validation", EVAL_HOPS, BASELINE_EVAL_SIZE, SEED + 1, unique_instances=True, balanced=True)
test_ids = store.select("test", EVAL_HOPS, TEST_EVAL_SIZE, SEED + 2, unique_instances=True, balanced=True) if TEST_EVAL_SIZE else []
assert not set(train_ids) & set(validation_ids)
assert not set(train_ids) & set(test_ids)
display(pd.DataFrame([{"subset": label, "questions": len(ids), "hops": dict(Counter(store.record(qid)["hop_count"] for qid in ids))}
                      for label, ids in [("train", train_ids), ("fixed validation", validation_ids), ("held-out test", test_ids)]]))
print(f"Graph: {len(graph.entities)} entities, {len(graph.triples)} stored triples; dataset valid: {dataset_validation['valid']}")
print("Selected train rows are real existing query instances; a configured maximum larger than the available pool is reported, not filled with invented rows.")
display(Markdown("**Example model-visible prompt (no gold program or answer metadata):**"))
display(pd.DataFrame(store.prompt(train_ids[0])))
private_probe = store.record(train_ids[0])
private_probe.update(answer_ids=["VERIFIER_PRIVATE_SENTINEL"], gold_entity_paths=[["VERIFIER_PRIVATE_SENTINEL"]], relation_path=["VERIFIER_PRIVATE_SENTINEL"])
assert public_prompt(graph, private_probe, MAX_TOOL_CALLS) == store.prompt(train_ids[0])
assert "VERIFIER_PRIVATE_SENTINEL" not in json.dumps(store.prompt(train_ids[0]))
fixed_selection = {"train_ids": train_ids, "validation_ids": validation_ids, "test_ids": test_ids}
(RUN_ROOT / "fixed_task_ids.json").write_text(json.dumps(fixed_selection, indent=2) + "\n")

,subset,questions,hops
0,train,668,"{2: 334, 3: 334}"
1,fixed validation,30,"{2: 10, 3: 10, 4: 10}"
2,held-out test,30,"{2: 10, 3: 10, 4: 10}"


Graph: 2000 entities, 5000 stored triples; dataset valid: True
Selected train rows are real existing query instances; a configured maximum larger than the available pool is reported, not filled with invented rows.


**Example model-visible prompt (no gold program or answer metadata):**

,role,content
0,system,Navigate the fictional research graph to answe...
1,user,Question: Which funding organizations sponsor ...


11718

<a id="tools"></a>
## 4. Stateful real graph environment

Every rollout creates a new GraphEnvironment and discovery-gated NavigationSession. Tools expose only discovered entity IDs, readable labels, types and relation operations. Invalid calls consume the same call budget as valid calls. Inverse traversal follows the existing ontology; one-to-many responses are complete. A response exceeding an explicit observation limit returns an error rather than silently hiding neighbors. The graph is shared read-only; discoveries, submissions, call counters and logs are isolated.

Submission ends the episode immediately. Its acknowledgment never reveals correctness. Plain-text final answers are not submissions. Malformed calls and token/context truncation are recorded, never repaired into fictional model actions.

The following **deterministic tool contract check is not a model trajectory or baseline**. It uses an existing reference path only to test graph operations and binary verification; gold programs are never supplied to a model.

In [5]:
fixture_id = store.select("train", [min(TRAIN_HOPS)], count=1, seed=SEED)[0]
fixture = store.record(fixture_id)
fixture_env = GraphEnvironment(graph, fixture["start_entity"], LIMITS)
fixture_path = fixture["gold_entity_paths"][0]
for head, relation, tail in zip(fixture_path, fixture["relation_path"], fixture_path[1:]):
    observation = fixture_env.get_neighbors(head, relation)
    assert observation["ok"] and tail in [entity["id"] for entity in observation["entities"]]
ack = fixture_env.submit_answer(fixture_path[-1])
assert fixture_env.done and fixture_env.termination_reason == "submitted"
assert "reward" not in ack and "correct" not in ack
assert store.reward(fixture_id, fixture_env.submitted_answer) == 1.0
assert store.reward(fixture_id, None) == 0.0
assert store.reward(fixture_id, fixture["start_entity"]) == 0.0
isolation_probe = GraphEnvironment(graph, fixture["start_entity"], LIMITS)
assert isolation_probe.calls == 0 and isolation_probe.submitted_answer is None
display(pd.DataFrame(fixture_env.trajectory))
print("Non-model tool contract checks passed. Schema tools:", [s["function"]["name"] for s in TOOL_SCHEMAS])

,index,name,arguments,observation
0,1,get_neighbors,"{'entity_id': 'researcher_000514', 'relation_i...","{'ok': True, 'entity_id': 'researcher_000514',..."
1,2,get_neighbors,"{'entity_id': 'university_000011', 'relation_i...","{'ok': True, 'entity_id': 'university_000011',..."
2,3,submit_answer,{'entity_id': 'city_000068'},"{'ok': True, 'submitted': 'city_000068', 'epis..."


Non-model tool contract checks passed. Schema tools: ['get_relations', 'get_neighbors', 'submit_answer']


<a id="model"></a>
## 5. Qwen3 initialization with NF4 QLoRA

The exact Hub revision is pinned. All seven Qwen attention/MLP projection suffixes are verified against the loaded architecture before applying rank-8, alpha-16 LoRA with dropout 0. Only LoRA parameters may require gradients. The base model is loaded once on one GPU, with double-quantized NF4, SDPA attention, and non-reentrant gradient checkpointing. Training caches are disabled. Generation uses one bounded KV cache at a time; rollouts are sampled sequentially rather than holding four inference batches in memory.

**Memory fallback, in order:** lower `MAX_TURN_TOKENS`/`MAX_COMPLETION_TOKENS`/context limits; lower tool interactions if the tasks still fit; reduce optional evaluation frequency and clear old trainer/optimizer objects; reduce `NUM_GENERATIONS` from 4 to 2 **with matching accumulation divisibility**. If still necessary, explicitly set `MODEL_NAME="Qwen/Qwen3-0.6B"` and rerun from model initialization. The notebook never switches model silently. Lowering the budget changes the task and should be reflected in a fresh baseline. With eight calls, a four-hop agent cannot list relations separately at every stage plus submit (that costs nine); it can reuse relation knowledge or you can explicitly increase the budget for both evaluations.
Generation explicitly uses the same native CUDA AMP dtype as the trainer, including the baseline before Accelerate wraps the model. KV caching is enabled only for bounded, gradient-free generation; training keeps `model.config.use_cache=False`.


In [6]:
from kg_rlvr.training.grpo import load_qlora, assert_training_stack
from kg_rlvr.training.rollouts import QwenChatCodec
model, tokenizer, COMPUTE_DTYPE = None, None, None
if RUN_MODEL:
    assert_training_stack(require_cuda=True)
    try:
        model, tokenizer, COMPUTE_DTYPE = load_qlora(MODEL_NAME, MODEL_REVISION, LORA_R, LORA_ALPHA)
    except torch.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("Model load ran out of memory. Apply the documented smaller-budget/model fallback explicitly; no replacement model has been loaded.")
    codec = QwenChatCodec(tokenizer)
    prompt_lengths = [len(codec.prompt_ids(store.prompt(qid))) for qid in set(train_ids + validation_ids + test_ids)]
    if max(prompt_lengths) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Largest selected prompt has {max(prompt_lengths)} tokens > configured {MAX_PROMPT_TOKENS}")
    format_probe = '<tool_call>\n{"name":"get_relations","arguments":{"entity_id":"' + fixture["start_entity"] + '"}}\n</tool_call><|im_end|>'
    from kg_rlvr.training.environment import parse_qwen_action
    assert parse_qwen_action(format_probe)[0] == "get_relations"
    feedback = codec.decode(codec.feedback_ids("get_relations", {"ok": True, "relations": ["works_at"]}))
    assert "<tool_response>" in feedback
    print("Model", MODEL_NAME, "revision", MODEL_REVISION, "dtype", COMPUTE_DTYPE, "max public prompt tokens", max(prompt_lengths))
    MEMORY.append(memory_snapshot("after_qlora_load"))
else:
    print("Qwen weights and NF4 initialization not run: no enabled CUDA model runtime.")

Qwen weights and NF4 initialization not run: no enabled CUDA model runtime.


<a id="baseline"></a>
## 6. Frozen base-model evaluation before training

The adapter is disabled for the baseline. The same starting prompts, graph tools, discovery rules, token limits and call budget are used after training. Evaluation uses greedy decoding and no gold guidance. CUDA kernels can still introduce platform-dependent nondeterminism. The default 30 validation questions provide about ten per hop; this is a small diagnostic sample, not a precise improvement estimate.

In [7]:
from kg_rlvr.training.rollouts import evaluate_policy
BASE_METRICS, BASE_TRACES = None, []
if RUN_MODEL and RUN_BASELINE:
    with model.disable_adapter():
        BASE_METRICS, BASE_TRACES = evaluate_policy(model, tokenizer, store, validation_ids, LIMITS, EVALUATION_DIR, "base_validation")
    display(pd.Series(BASE_METRICS, name="measured frozen-base baseline").to_frame())
    MEMORY.append(memory_snapshot("after_baseline"))
    if not any(event["name"] == "get_neighbors" and event["observation"].get("ok") for trace in BASE_TRACES for event in trace["tool_events"]):
        raise RuntimeError("The base evaluation executed no valid graph-neighbor tools. Inspect real trajectories and tool formatting before training; no working model tool use is claimed.")
else:
    print("Baseline model evaluation not run; no baseline accuracy is reported.")

Baseline model evaluation not run; no baseline accuracy is reported.


<a id="grpo"></a>
## 7. GRPO configuration and rollout integration

The native TRL 1.8.0 environment loop was inspected. It does not consult per-episode completion after submission, so this notebook uses its supported **custom `rollout_func`** instead. The deviation is orchestration only: `GRPOTrainer` still computes group-normalized advantages, the GRPO clipped objective, gradients, and optimizer steps.

- TRL 1.8.0's actual RepeatSampler already supplies `G` consecutive copies of each prompt to this hook. We assert identical prompts and fresh state within every group and **do not repeat them again**. The version is pinned because this experimental API can change.
- Raw sampled action IDs are preserved, including EOS. Tool suffixes are rendered with the real Qwen template and concatenated, without re-tokenizing or canonicalizing generated actions. `env_mask=1` marks model tokens; `0` marks observations and injected assistant prefixes. TRL consumes this mask as its tool loss mask. Observations remain attention context but are excluded from the policy objective.
- The custom hook returns `logprobs=None` as supported by the inspected release; TRL computes on-policy log-probabilities. Sampling uses temperature 0.8, no top-k/p warping, and fresh GenerationConfig defaults, matching the trainer's temperature-scaled policy probabilities.
- One GPU × microbatch 1 × accumulation 4 = four completion trajectories per optimizer update. With `G=4`, that is **one unique question/group per update**. Accumulation 8 gives two groups per update. Generation batch size equals accumulation; `num_iterations=1` keeps updates on-policy.
- AdamW updates only the small FP32 LoRA parameters, with learning rate 5e-6, no weight decay, constant scheduling and gradient norm cap 1. There is **no KL reference model** (`beta=0`) and no KL metric. This avoids an unnecessary base-model copy. Zero-reward or all-success groups have zero reward variance and zero policy signal; both are logged.

No native `tools` or `environment_factory` argument is passed alongside the custom hook, which would execute calls a second time. Unsupported legacy config arguments are rejected. Periodic evaluation is a separate greedy callback, so GRPO generation-group batching is not confused with evaluation batching.

In [8]:
from kg_rlvr.training.grpo import (
    build_trainer, lora_snapshot, verify_optimizer_update, training_manifest,
    check_resume, export_adapter, reload_adapter_in_place, observed_trajectory_html,
)
EXPERIMENT_CONFIG = {"model_name": MODEL_NAME, "revision": MODEL_REVISION,
    "train_hops": TRAIN_HOPS, "eval_hops": EVAL_HOPS, "num_generations": NUM_GENERATIONS,
    "microbatch": 1, "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "unique_questions_per_update": GRADIENT_ACCUMULATION_STEPS // NUM_GENERATIONS,
    "learning_rate": LEARNING_RATE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "lora_dropout": 0.0, "limits": asdict(LIMITS), "smoke_steps": SMOKE_TEST_STEPS,
    "full_steps": INITIAL_TRAINING_STEPS, "save_steps": SAVE_STEPS,
    "evaluate_every_steps": EVALUATE_EVERY_STEPS, "seed": SEED}
display(pd.Series(EXPERIMENT_CONFIG, name="requested configuration").to_frame())
(RUN_ROOT / "experiment_config.json").write_text(json.dumps(EXPERIMENT_CONFIG, indent=2) + "\n")
smoke_trainer, full_trainer, final_trainer = None, None, None
SMOKE_RESULT, FULL_RESULT = None, None

,requested configuration
model_name,Qwen/Qwen3-1.7B
revision,70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
train_hops,"[2, 3]"
eval_hops,"[2, 3, 4]"
num_generations,4
microbatch,1
gradient_accumulation,4
unique_questions_per_update,1
learning_rate,0.000005
lora_r,8


<a id="smoke"></a>
## 8. Genuine two-step GRPO smoke test

This is not an inference-only test. The real model samples multiple independent episodes, calls graph tools, receives binary rewards, and runs `trainer.train()`. The gate requires finite logged losses, a positive finite gradient norm, changed trainable LoRA tensors, a saved adapter plus optimizer/scheduler/trainer/RNG checkpoint, and measured peak GPU memory.

The smoke pool is a seeded sample from the shortest configured **training** hop category, to keep initial execution bounded. Full training uses the full selected training pool. No scripted actions or reference programs guide model rollouts.

If no LoRA tensor changes, the smoke fails even if inference and optimizer plumbing ran. Inspect zero-variance groups, raw actions, malformed calls, submission rate, and truncation. With the requested sparse binary reward, homogeneous groups legitimately provide no gradient. Increase real smoke groups or adjust explicit limits/sampling after diagnosis; no reward shaping or manufactured successes are inserted. Full training must not proceed after failure.

In [9]:
if RUN_MODEL and RUN_SMOKE_TEST:
    smoke_dir = CHECKPOINT_ROOT / "smoke"
    if smoke_dir.exists() and any(smoke_dir.iterdir()):
        raise FileExistsError("Smoke output already exists. Choose a fresh RUN_NAME to avoid mixing checkpoints and measured logs.")
    smoke_ids = store.select("train", [min(TRAIN_HOPS)], count=max(8, GRADIENT_ACCUMULATION_STEPS // NUM_GENERATIONS * SMOKE_TEST_STEPS), seed=SEED)
    smoke_manifest = training_manifest(MODEL_NAME, MODEL_REVISION, LIMITS, smoke_ids, NUM_GENERATIONS,
        GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, SEED, LORA_R, LORA_ALPHA)
    smoke_manifest["source_sha256"] = INPUT_SHA256
    smoke_dir.mkdir(parents=True, exist_ok=True)
    (smoke_dir / "run_manifest.json").write_text(json.dumps(smoke_manifest, indent=2) + "\n")
    smoke_trainer, smoke_rollout = build_trainer(model, tokenizer, store, smoke_ids, LIMITS, smoke_dir, "smoke",
        SMOKE_TEST_STEPS, NUM_GENERATIONS, GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, SMOKE_TEST_STEPS, SEED)
    before = lora_snapshot(model)
    torch.cuda.reset_peak_memory_stats()
    try:
        smoke_train_result = smoke_trainer.train()
        SMOKE_RESULT = verify_optimizer_update(model, before, smoke_trainer, smoke_dir, SMOKE_TEST_STEPS)
    except Exception as error:
        (RUN_ROOT / "smoke_failure.json").write_text(json.dumps({"error": repr(error), "gpu": memory_snapshot("smoke_failure"), "configuration": EXPERIMENT_CONFIG}, indent=2) + "\n")
        raise  # no fake reward/loss output and no automatic full-training fallback
    finally:
        del before
        gc.collect()
    smoke_events = [event for line in (smoke_dir / "trajectories.jsonl").read_text().splitlines() for event in json.loads(line)["tool_events"]]
    assert any(e["name"] == "get_neighbors" and e["observation"].get("ok") for e in smoke_events)
    (RUN_ROOT / "smoke_verified.json").write_text(json.dumps(SMOKE_RESULT, indent=2) + "\n")
    MEMORY.append(memory_snapshot("after_verified_smoke"))
    final_trainer = smoke_trainer
    display(pd.Series(SMOKE_RESULT, name="measured genuine GRPO smoke result").to_frame())
else:
    print("GPU GRPO smoke test not executed. No genuine optimizer update or weight change has been verified in this run.")

GPU GRPO smoke test not executed. No genuine optimizer update or weight change has been verified in this run.


<a id="full"></a>
## 9. Full training and checkpoint resumption

Enable `RUN_FULL_TRAINING=True` to use the configured real training pool for the initial 30-step run. The smoke adapter is the warm start, while this phase intentionally creates a fresh optimizer/scheduler. Its baseline comparison therefore includes smoke updates. When resuming, the full checkpoint overrides that warm start with its saved adapter, optimizer, scheduler, RNG and trainer state.

`INITIAL_TRAINING_STEPS` is the Trainer's **absolute full-phase optimizer-step target**, not additional steps after a resume. `RESUME_FROM_CHECKPOINT` must point to a complete `checkpoint-*` directory accompanied by the original `run_manifest.json`. Adapter-only exports are inference artifacts and do not provide exact resumption. The manifest checks source fingerprints, selected training IDs, model/revision, versions, LoRA and rollout settings. Extending the target is supported by the constant scheduler; GPU kernels/session differences can still prevent bitwise-identical continuation.

Checkpoints are saved under `/kaggle/working/checkpoints/<RUN_NAME>/full/`, retaining two. A non-resume run refuses to mix logs with an earlier run. Periodic validation frequency is configurable and initially disabled for runtime/memory economy.
Set full training before Run All when ready; the smoke gate still runs first. After a completed smoke-only Run All, restart with a fresh `RUN_NAME` and full training enabled. Export/reload has already released the old trainers and made that adapter inference-only. To continue within the initial session, stop after section 8, set `RUN_FULL_TRAINING=True`, then execute sections 9 onward before export.


In [10]:
if RUN_MODEL and RUN_FULL_TRAINING:
    if SMOKE_RESULT is None:
        raise RuntimeError("Full training requires this run's verified genuine smoke update. Enable and complete RUN_SMOKE_TEST first.")
    if smoke_trainer is None:
        raise RuntimeError("Full training must follow smoke before export/reload. For a completed smoke-only notebook, restart with a fresh RUN_NAME and RUN_FULL_TRAINING=True, then Run All.")
    full_dir = CHECKPOINT_ROOT / "full"
    full_manifest = training_manifest(MODEL_NAME, MODEL_REVISION, LIMITS, train_ids, NUM_GENERATIONS,
        GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, SEED, LORA_R, LORA_ALPHA)
    full_manifest["source_sha256"] = INPUT_SHA256
    resume_path = check_resume(RESUME_FROM_CHECKPOINT, full_manifest) if RESUME_FROM_CHECKPOINT else None
    if not resume_path and full_dir.exists() and any(full_dir.iterdir()):
        raise FileExistsError("Full-training output exists: use a new RUN_NAME or an explicit compatible resume checkpoint.")
    if resume_path:
        saved_step = json.loads((Path(resume_path) / "trainer_state.json").read_text())["global_step"]
        if saved_step >= INITIAL_TRAINING_STEPS:
            raise ValueError("Resume checkpoint has already reached the configured absolute step target; increase INITIAL_TRAINING_STEPS.")
    full_dir.mkdir(parents=True, exist_ok=True)
    (full_dir / "run_manifest.json").write_text(json.dumps(full_manifest, indent=2) + "\n")
    # Release smoke optimizer state before allocating the full-phase optimizer.
    smoke_trainer.optimizer = None
    smoke_trainer.lr_scheduler = None
    smoke_trainer.accelerator.free_memory()
    smoke_trainer = final_trainer = None
    gc.collect(); torch.cuda.empty_cache()
    full_trainer, full_rollout = build_trainer(model, tokenizer, store, train_ids, LIMITS, full_dir, "full",
        INITIAL_TRAINING_STEPS, NUM_GENERATIONS, GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, SAVE_STEPS, SEED,
        eval_task_ids=validation_ids[:PERIODIC_EVAL_SIZE], eval_every=EVALUATE_EVERY_STEPS)
    torch.cuda.reset_peak_memory_stats()
    result = full_trainer.train(resume_from_checkpoint=resume_path)
    FULL_RESULT = {"global_step": full_trainer.state.global_step, "train_loss": float(result.training_loss)}
    if not math.isfinite(FULL_RESULT["train_loss"]):
        raise FloatingPointError("Non-finite full-phase training loss")
    final_trainer = full_trainer
    MEMORY.append(memory_snapshot("after_full_training"))
    (RUN_ROOT / "full_training_result.json").write_text(json.dumps(FULL_RESULT, indent=2) + "\n")
    display(pd.Series(FULL_RESULT, name="measured full-training result").to_frame())
else:
    print("Full training not run. Set RUN_FULL_TRAINING=True for an explicit longer experiment after the smoke gate.")

Full training not run. Set RUN_FULL_TRAINING=True for an explicit longer experiment after the smoke gate.


<a id="metrics"></a>
## 10. Actual rollout, optimizer, and resource metrics

Reward and success plots are read from real rollout JSONL records. Zero-reward groups and all-homogeneous groups are separate: both all-zero and all-one groups have zero variance. Call rates count attempted calls; malformed calls are invalid attempts. Episodes with no calls have a valid-call rate of zero by definition. Completion here means a submitted entity, with other termination reasons tabulated separately. Gold reward is answer accuracy under the single-answer task.

KL divergence is unavailable because `beta=0`. No exploration cost or GPU measurement is fabricated for a skipped phase. Wall time includes actual rollout/forward/backward work. Baseline/trained trajectories and tool observations are private diagnostics, not model prompt columns.

In [11]:
def read_rows(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []
phase_dirs = [CHECKPOINT_ROOT / phase for phase in ["smoke", "full"]]
rollout_rows = [row for directory in phase_dirs for row in read_rows(directory / "rollout_metrics.jsonl")]
trainer_rows = [{"phase": directory.name, **row} for directory in phase_dirs for row in read_rows(directory / "trainer_metrics.jsonl")]
step_rows = [{"phase": directory.name, **row} for directory in phase_dirs for row in read_rows(directory / "step_metrics.jsonl")]
rollout_metrics, trainer_metrics, step_metrics = map(pd.DataFrame, [rollout_rows, trainer_rows, step_rows])
if rollout_rows:
    display(rollout_metrics[["phase", "step", "mean_episode_reward", "zero_reward_group_fraction", "zero_variance_group_fraction", "average_tool_calls", "invalid_tool_call_rate", "submission_rate"]])
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .2})
    figure_dir = RUN_ROOT / "figures"; figure_dir.mkdir(exist_ok=True)
    def plot_measured(ax, frame, column, title, ylabel):
        if frame.empty or column not in frame or frame[column].dropna().empty:
            ax.text(.5, .5, "Not measured", transform=ax.transAxes, ha="center")
        else:
            for phase, group in frame.groupby("phase", sort=False):
                subset = group.dropna(subset=[column])
                ax.plot(subset.step, subset[column], marker=".", label=phase)
            ax.legend()
        ax.set(title=title, xlabel="Optimizer step (within phase)", ylabel=ylabel)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for ax, column, title in zip(axes.flat,
        ["mean_episode_reward", "zero_reward_group_fraction", "zero_variance_group_fraction", "average_tool_calls", "invalid_tool_call_rate", "submission_rate"],
        ["Actual mean episode reward / success", "All-zero reward groups", "Zero-variance comparison groups", "Tool calls per episode", "Invalid tool-call fraction", "Answer submission fraction"]):
        plot_measured(ax, rollout_metrics, column, title, "Calls" if column == "average_tool_calls" else "Fraction / mean reward")
    fig.savefig(figure_dir / "rollout_metrics.png", bbox_inches="tight"); plt.show(); plt.close(fig)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for ax, frame, column, title, ylabel in [
        (axes[0,0], trainer_metrics, "loss", "Actual GRPO loss", "Loss"),
        (axes[0,1], trainer_metrics, "grad_norm", "Gradient norm", "Norm"),
        (axes[0,2], step_metrics, "step_seconds", "Measured optimizer step duration", "Seconds"),
        (axes[1,0], step_metrics, "allocated_gib", "CUDA allocated memory", "GiB"),
        (axes[1,1], step_metrics, "peak_allocated_gib", "CUDA peak allocated within phase", "GiB"),
        (axes[1,2], rollout_metrics, "episodes_all_calls_valid_rate", "Episodes with only valid calls", "Fraction")]:
        plot_measured(ax, frame, column, title, ylabel)
    fig.savefig(figure_dir / "optimizer_metrics.png", bbox_inches="tight"); plt.show(); plt.close(fig)
    rollout_metrics.to_json(RUN_ROOT / "rollout_metrics.jsonl", orient="records", lines=True)
    trainer_metrics.to_json(RUN_ROOT / "trainer_metrics.jsonl", orient="records", lines=True)
    step_metrics.to_json(RUN_ROOT / "step_metrics.jsonl", orient="records", lines=True)
    training_traces = [row for directory in phase_dirs for row in read_rows(directory / "trajectories.jsonl")]
    (RUN_ROOT / "training_trajectories.jsonl").write_text("".join(json.dumps(t) + "\n" for t in training_traces))
else:
    print("No model rollouts or optimizer metrics exist in this run. Reward/loss charts are not fabricated.")
display(pd.DataFrame(MEMORY))

No model rollouts or optimizer metrics exist in this run. Reward/loss charts are not fabricated.


,stage,system_available_gib,cuda_available,allocated_gib,reserved_gib,peak_allocated_gib,peak_reserved_gib
0,before_model,0.756943,False,None,None,None,None


<a id="evaluation"></a>
## 11. Post-training validation and separate test evaluation

These are real greedy tool-use episodes with the exact same runtime limits as the baseline. The validation ID list is reused verbatim; the test subset is separate and never used for updates. Comparisons are descriptive at this sample size. No improvement is claimed unless actual measurements support it, and a small difference alone is not statistical evidence.

In [12]:
POST_METRICS, TEST_METRICS, POST_TRACES, TEST_TRACES = None, None, [], []
if RUN_MODEL and SMOKE_RESULT is not None:
    model = final_trainer.accelerator.unwrap_model(final_trainer.model)
    POST_METRICS, POST_TRACES = evaluate_policy(model, tokenizer, store, validation_ids, LIMITS, EVALUATION_DIR, "trained_validation")
    if test_ids:
        TEST_METRICS, TEST_TRACES = evaluate_policy(model, tokenizer, store, test_ids, LIMITS, EVALUATION_DIR, "trained_test")
    comparison_rows = []
    for hop in EVAL_HOPS:
        comparison_rows.append({"metric": f"{hop}-hop validation accuracy", "base": (BASE_METRICS or {}).get("accuracy_by_hop", {}).get(str(hop)), "GRPO-trained": POST_METRICS["accuracy_by_hop"].get(str(hop))})
    for metric in ["accuracy", "valid_tool_call_rate", "episodes_with_valid_calls_rate", "average_tool_calls", "average_tool_calls_correct", "submission_rate"]:
        comparison_rows.append({"metric": metric, "base": (BASE_METRICS or {}).get(metric), "GRPO-trained": POST_METRICS.get(metric)})
    comparison = pd.DataFrame(comparison_rows)
    display(comparison)
    print("Trained validation termination reasons:", POST_METRICS["termination_reasons"])
    print("Separate held-out test metrics:", TEST_METRICS)
    comparison.to_json(EVALUATION_DIR / "base_vs_trained.json", orient="records", indent=2)
    MEMORY.append(memory_snapshot("after_post_training_evaluation"))
else:
    print("Post-training model evaluation not run. No trained-model accuracy or improvement claim is available.")

Post-training model evaluation not run. No trained-model accuracy or improvement claim is available.


<a id="export"></a>
## 12. Adapter export, reload, and download

The adapter-only export goes to `/kaggle/working/grpo_adapter/`, with tokenizer files, immutable model revision, training configuration, dependency versions and references to actual logs/evaluations. There is no unnecessary full base-model copy. Full trainer checkpoints remain separate under `checkpoints/` and include optimizer/scheduler/RNG state.

Before the reload test, optimizer/trainer references are released. The adapter is unloaded and reloaded from disk onto the **same quantized base weights**. A fresh graph-tool evaluation of three fixed validation questions checks the reloaded policy. The ZIP includes adapter and run diagnostics, excluding large trainer checkpoints and Hub caches. Save checkpoints separately if you intend to resume.

For later inference, load the same model/revision using `AutoModelForCausalLM.from_pretrained(..., quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16), device_map={"":0})`, load tokenizer files from the adapter directory, then call `PeftModel.from_pretrained(base, adapter_directory, is_trainable=False)`. The exported manifest records the exact revision and limits. To reproduce graph interaction, use this repository's `evaluate_policy`/rollout engine with that policy and the saved graph.

In [13]:
RELOAD_METRICS, RELOAD_TRACES = None, []
if RUN_MODEL and SMOKE_RESULT is not None:
    manifest = full_manifest if FULL_RESULT is not None else smoke_manifest
    export_adapter(model, tokenizer, ADAPTER_DIR, manifest, extra={"experiment_config": EXPERIMENT_CONFIG,
        "final_phase": "full" if FULL_RESULT is not None else "smoke", "smoke_verified": SMOKE_RESULT,
        "full_training_result": FULL_RESULT, "hardware": hardware,
        "validation_metrics": POST_METRICS, "test_metrics": TEST_METRICS,
        "training_logs_directory": str(CHECKPOINT_ROOT), "evaluation_directory": str(EVALUATION_DIR)})
    for trainer in [smoke_trainer, full_trainer]:
        if trainer is not None:
            trainer.optimizer = None; trainer.lr_scheduler = None
            trainer.accelerator.free_memory()
    smoke_trainer = full_trainer = final_trainer = None
    if "trainer" in globals():
        del trainer
    gc.collect(); torch.cuda.empty_cache()
    model = reload_adapter_in_place(model, ADAPTER_DIR)
    RELOAD_METRICS, RELOAD_TRACES = evaluate_policy(model, tokenizer, store, validation_ids[:RELOAD_EVAL_SIZE], LIMITS, EVALUATION_DIR, "reloaded_adapter")
    display(pd.Series(RELOAD_METRICS, name="measured adapter reload evaluation").to_frame())
    MEMORY.append(memory_snapshot("after_adapter_reload_evaluation"))
    print("Adapter-only export:", ADAPTER_DIR, "Full checkpoints:", CHECKPOINT_ROOT)
else:
    print("No trained adapter exported or reloaded: GPU smoke training has not been verified.")

No trained adapter exported or reloaded: GPU smoke training has not been verified.


<a id="trajectories"></a>
## 13. Actual model trajectories and run evidence

The examples below come only from real model inference. No deterministic reference path is passed off as a model trace. Each example shows the original question/start, generated assistant tokens, executed calls, observations, submitted entity, private correct IDs, binary reward, call count, and termination reason. `observation_in_context=False` means that an acknowledgment/over-budget result was not fed into another model generation.

The optional interactive view contains **model-observed neighbor results only**. It does not highlight a gold reasoning path; an amber submitted node can be wrong. Training itself does not depend on this visualization.

In [14]:
actual_examples = []
for label, traces in [("base", BASE_TRACES), ("GRPO-trained", POST_TRACES), ("reloaded adapter", RELOAD_TRACES)]:
    if traces:
        actual_examples.append((label, max(traces, key=lambda t: (t["reward"], t["tool_calls"]))))
if len(actual_examples) < 3:
    pool = [("base", t) for t in BASE_TRACES] + [("GRPO-trained", t) for t in POST_TRACES]
    for item in pool:
        if len(actual_examples) >= 3:
            break
        if item not in actual_examples:
            actual_examples.append(item)
for label, trace in actual_examples[:3]:
    display(Markdown(f"### {label} · {trace['task_id']} · {trace['hop_count']} hops\n\n{trace['question']}"))
    display(pd.Series({"starting_entity": trace["start_entity"], "starting_label": graph.entities[trace["start_entity"]]["name"],
        "submitted_answer": trace["submitted_answer"], "gold_answer_ids": trace["gold_answer_ids"],
        "reward": trace["reward"], "tool_calls": trace["tool_calls"], "termination_reason": trace["termination_reason"]}).to_frame("actual outcome"))
    display(pd.DataFrame(trace["turns"]))
if actual_examples:
    view_path = RUN_ROOT / "observed_model_trajectory.html"
    observed_trajectory_html(graph, actual_examples[-1][1], view_path)
    print("Standalone model-observed graph view:", view_path)
    # Relative links can depend on notebook-server/Kaggle trust policy; the exported file is independent.
else:
    print("No actual model-generated trajectories are available in this CPU-only run; none are fabricated.")

assert fingerprint_inputs(DATA_ROOT) == INPUT_SHA256, "The source dataset changed"
peaks = [m["peak_allocated_gib"] for m in MEMORY if m["peak_allocated_gib"] is not None]
evidence = {"runtime_identifies_as_kaggle": IS_KAGGLE, "cuda_available": HAS_CUDA,
    "model_name": MODEL_NAME, "model_revision": MODEL_REVISION, "configuration": EXPERIMENT_CONFIG,
    "installed_versions": dependency_versions(), "hardware": hardware,
    "dataset_valid": dataset_validation["valid"], "source_files_unchanged": True,
    "source_sha256": INPUT_SHA256, "graph_tool_contract_check_passed": True,
    "base_model_evaluation_performed": BASE_METRICS is not None,
    "genuine_grpo_update_verified": SMOKE_RESULT is not None,
    "smoke_result": SMOKE_RESULT, "full_training_result": FULL_RESULT,
    "adapter_reload_evaluated": RELOAD_METRICS is not None,
    "peak_measured_gpu_allocated_gib": max(peaks) if peaks else None,
    "real_model_trajectory_examples": len(actual_examples[:3]),
    "gpu_dependent_checks_status": "verified smoke gate; inspect measured results" if SMOKE_RESULT is not None else "UNVERIFIED — no successful CUDA training smoke in this run"}
(RUN_ROOT / "run_evidence.json").write_text(json.dumps(evidence, indent=2) + "\n")
(RUN_ROOT / "memory_snapshots.json").write_text(json.dumps(MEMORY, indent=2) + "\n")
display(pd.Series({key: evidence[key] for key in ["dataset_valid", "source_files_unchanged", "base_model_evaluation_performed", "genuine_grpo_update_verified", "adapter_reload_evaluated", "peak_measured_gpu_allocated_gib", "real_model_trajectory_examples", "gpu_dependent_checks_status"]}, name="actual execution evidence").to_frame())
print("Run artifacts:", RUN_ROOT)

if RUN_MODEL and SMOKE_RESULT is not None:
    if MAKE_ZIP:
        import zipfile
        zip_path = WORK_ROOT / f"{RUN_NAME}_adapter_and_diagnostics.zip"
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for folder in [ADAPTER_DIR, RUN_ROOT]:
                for file in sorted(folder.rglob("*")):
                    if file.is_file():
                        archive.write(file, file.relative_to(WORK_ROOT))
        print("Download from Kaggle output panel:", zip_path)


No actual model-generated trajectories are available in this CPU-only run; none are fabricated.


,actual execution evidence
dataset_valid,True
source_files_unchanged,True
base_model_evaluation_performed,False
genuine_grpo_update_verified,False
adapter_reload_evaluated,False
peak_measured_gpu_allocated_gib,None
real_model_trajectory_examples,0
gpu_dependent_checks_status,UNVERIFIED — no successful CUDA training smoke...


Run artifacts: /Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo/runs/kg_grpo_seed41
